# 📱 Spam SMS Detector

**Goal:** build a filter that reads a text message and decides whether it's **spam** or a normal message (**"ham"**).

**New skills in this project**
- Turning **text into numbers** a model can use: bag-of-words and **TF-IDF**
- **Naive Bayes**, the classic (and still great) text classifier
- Dealing with **imbalanced classes** — only ~13% of messages are spam
- Reading what the model learned (which words scream "spam")

⏱️ One weekend. 
**How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

## 1. Load the data
The SMS Spam Collection: 5,572 real text messages from the UK and Singapore, each labelled `ham` or `spam`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])
print(sms.shape)
sms.head(10)

In [ ]:
print(sms["label"].value_counts())
print()
print(sms["label"].value_counts(normalize=True).round(3))

⚠️ **Imbalanced classes:** 87% ham, 13% spam. A "model" that says *ham* for everything is 87% accurate and catches zero spam. So we'll focus on **precision** and **recall** for the spam class (you met these in the Titanic project).

## 2. Explore — what does spam look like?

In [ ]:
for msg in sms[sms["label"] == "spam"]["message"].sample(5, random_state=1):
    print("🚫", msg, "\n")

In [ ]:
sms["length"] = sms["message"].str.len()
sms.groupby("label")["length"].describe().round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for label in ["ham", "spam"]:
    sms[sms["label"] == label]["length"].plot(kind="hist", bins=60, alpha=0.6, label=label, ax=ax, range=(0, 300))
ax.set_xlabel("message length (characters)"); ax.legend()
plt.show()

Spam messages tend to be longer (they pack in offers, numbers and links). Spam also loves words like *FREE*, *win*, *claim*, *prize*, *txt*.

## 3. Turning text into numbers

Models need numbers. The simplest idea is **bag-of-words**: build a vocabulary of every word, then represent each message by *how many times each word appears*. Word order is thrown away — hence "bag".

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

tiny = ["win a free prize now", "are we meeting now", "free free free"]
vec = CountVectorizer()
counts = vec.fit_transform(tiny)
pd.DataFrame(counts.toarray(), columns=vec.get_feature_names_out(), index=tiny)

### TF-IDF — smarter word weights

Some words (*the*, *you*, *to*) appear everywhere and tell us nothing. **TF-IDF** (term frequency × inverse document frequency) gives a word a **high score if it's common in this message but rare across all messages**. Distinctive words get boosted; filler words get shrunk.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
weights = tfidf.fit_transform(tiny)
pd.DataFrame(weights.toarray().round(2), columns=tfidf.get_feature_names_out(), index=tiny)

Notice `now` (in two messages) gets a lower weight than `prize` (in one).

## 4. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X = sms["message"]
y = (sms["label"] == "spam").astype(int)   # 1 = spam, 0 = ham
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "train /", len(X_test), "test; spam in test:", y_test.sum())

## 5. Naive Bayes

**Naive Bayes** asks: *"Given the words in this message, is it more likely to be spam or ham?"* It learns how often each word appears in spam vs ham, then combines those clues. It's "naive" because it assumes words are independent (they're not, but it works remarkably well). It trains in milliseconds.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

nb_model = make_pipeline(TfidfVectorizer(), MultinomialNB())
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
print(classification_report(y_test, nb_pred, target_names=["ham", "spam"], digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, nb_pred, display_labels=["ham", "spam"], cmap="Blues")
plt.title("Naive Bayes")
plt.show()

Look at the spam row: **precision is ~100%** (it almost never flags a real message) but **recall is lower** — some spam slips through. For a spam filter that's the right *direction* (losing a real message is worse than seeing a spam), but we can do better.

## 6. Handling the imbalance

Three common tricks:
1. **Better features** — include pairs of words (*"free entry"*, *"call now"*) with `ngram_range=(1, 2)`.
2. **Class weights** — tell the model mistakes on the rare class cost more (`class_weight="balanced"`).
3. **Move the threshold** — flag as spam at a lower probability.

Let's try a **logistic regression** with class weights and word pairs:

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(class_weight="balanced", C=10, max_iter=2000),
)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
print(classification_report(y_test, lr_pred, target_names=["ham", "spam"], digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, display_labels=["ham", "spam"], cmap="Greens")
plt.title("Logistic regression (balanced, 1-2 word features)")
plt.show()

## 7. Comparing fairly with cross-validation

We'll use **F1 for the spam class** as our single score — it balances precision and recall, and ignores the easy "ham" majority.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC

candidates = {
    "Naive Bayes": make_pipeline(TfidfVectorizer(), MultinomialNB()),
    "Naive Bayes (tuned)": make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), MultinomialNB(alpha=0.05)),
    "Logistic regression (balanced)": lr_model,
    "Linear SVM (balanced)": make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                                           LinearSVC(class_weight="balanced")),
}
for name, m in candidates.items():
    s = cross_val_score(m, X_train, y_train, cv=5, scoring="f1")
    print(f"{name:32s} spam F1 = {s.mean():.3f}")

## 8. What did the model learn?
Logistic regression gives every word (or word pair) a weight. The biggest positive weights are the strongest *spam* signals:

In [ ]:
vec = lr_model[0]
clf = lr_model[-1]
words = vec.get_feature_names_out()
coefs = clf.coef_[0]
top = np.argsort(coefs)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
pd.Series(coefs[top[-15:]], index=words[top[-15:]]).plot(kind="barh", ax=axes[0], color="tab:red", title="Strongest SPAM words")
pd.Series(coefs[top[:15]], index=words[top[:15]]).plot(kind="barh", ax=axes[1], color="tab:green", title="Strongest HAM words")
plt.tight_layout(); plt.show()

## 9. Look at the mistakes
Reading errors is one of the most useful habits in ML — it tells you what to fix next.

In [ ]:
errors = pd.DataFrame({"message": X_test, "actual": y_test, "predicted": lr_pred})
errors = errors[errors["actual"] != errors["predicted"]]
for _, r in errors.iterrows():
    kind = "MISSED SPAM" if r["actual"] == 1 else "FALSE ALARM"
    print(f"[{kind}] {r['message']}\n")

## 10. Try your own messages

In [ ]:
def check(message):
    p = lr_model.predict_proba([message])[0, 1]
    verdict = "🚫 SPAM" if p >= 0.5 else "✅ ham"
    print(f"{verdict}  ({p:.0%} spam)  —  {message}")

check("Congratulations! You've won a FREE iPhone. Call 09061701461 to claim now!")
check("Hey, are we still on for dinner tonight?")
check("URGENT: your account has been suspended. Reply with your PIN to reactivate")
check("Can you send me the notes from today's class?")

## 11. Save the model

In [ ]:
import joblib
joblib.dump(lr_model, "spam_model.joblib")
print("Saved spam_model.joblib — load it later with joblib.load('spam_model.joblib')")

## ✅ What you learned
- **Bag-of-words** and **TF-IDF** turn text into numbers.
- **Naive Bayes** is a fast, strong baseline for text.
- With **imbalanced classes**, judge models by precision/recall/F1 on the rare class, and use **class weights** or thresholds to rebalance.
- Reading a model's **weights** and **mistakes** tells you what it learned and what to fix.

## ✍️ Try it yourself
1. Add a `length` feature — does it help? (Hint: harder — look up `ColumnTransformer` with a text column and a numeric column.)
2. Try `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))` — character pieces instead of words. It catches tricks like "FR33".
3. Collect 20 SMS messages from your own phone (including spam!) and test the model. Does it work on Indian spam messages? Why or why not?
4. Wrap `check()` in a tiny web app with Gradio (`pip install gradio`, then `gradio.Interface(fn=..., inputs="text", outputs="text").launch()`).

In [ ]:
# Your code here